In [1]:
%load_ext autoreload
%autoreload 2

from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer, LlamaForCausalLM
from tqdm import tqdm
from datasets import load_dataset
import functools
import pickle

from selfcheckgpt.modeling_selfcheck import SelfCheckMQAG, SelfCheckBERTScore, SelfCheckNgram
from sklearn.metrics import roc_auc_score
import statistics
import spacy

from result_collector import trex_data_to_question_template, answer_trivia, answer_trex, load_data, model_dir

import torch
import numpy as np

In [2]:
org="openlm-research"
model_name = "open_llama_7b"
repo = f"{org}/{model_name}"

# Data related params
dataset_name = "trivia_qa"

# GPU
gpu = "1"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")

# SelfCheckGPT
self_checkgpt_temperature = 1.0
selfcheckgpt_n_trials = 20

In [3]:
dataset = load_data(dataset_name)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

100%|██████████| 138384/138384 [00:14<00:00, 9375.31it/s] 


In [4]:
tokenizer = LlamaTokenizer.from_pretrained(repo)
model = LlamaForCausalLM.from_pretrained(repo, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

In [5]:
selfcheck_bertscore = SelfCheckBERTScore(rescale_with_baseline=True)
selfcheck_ngram = SelfCheckNgram(n=1) # n=1 means Unigram, n=2 means Bigram, etc.

SelfCheck-BERTScore initialized
SelfCheck-1gram initialized


In [12]:
def generate_responses(question, str_response, tokenizer):

    # generate several responses to the question and (self)check them against the zero temp response
    inputs = tokenizer(question, return_tensors="pt").input_ids.to(device)
    start_pos = inputs.size(dim=-1)

    hitemp_str_responses = []
    for i in range(0, selfcheckgpt_n_trials):
        model_outputs = model.generate(
            inputs, do_sample=True, temperature=self_checkgpt_temperature, max_new_tokens=100, return_dict_in_generate=True, output_scores=True
        )
        generated_tokens_ids = model_outputs.sequences[0]
        generated_text = tokenizer.decode(generated_tokens_ids[start_pos:]).replace("\n", " ").strip()
        if generated_text:  # Only add non-empty responses
            hitemp_str_responses.append(generated_text)

    # Handle case where no valid responses were generated
    if not hitemp_str_responses:
        print(f"Warning: No valid responses generated for question: {question}")
        # Return empty results to skip this sample
        return [], [], [], []

    selfcheck_scores_bert_overall = []
    selfcheck_scores_bert_average = []
    selfcheck_ngram_overall = []
    
    sentences = [str_response]
    overall_bertscore = selfcheck_bertscore.predict(
        sentences = sentences,                          # list of sentences
        sampled_passages = hitemp_str_responses, # list of sampled passages
    )
    selfcheck_scores_bert_overall.append(overall_bertscore[0])
    
    nlp = spacy.load("en_core_web_sm")
    sentences = [sent for sent in nlp(str_response).sents]
    sentences = [sent.text.strip() for sent in sentences if len(sent) > 3]
    
    # Handle case where response has no valid sentences
    if sentences:
        all_bertscores = selfcheck_bertscore.predict(
            sentences = sentences,                          # list of sentences
            sampled_passages = hitemp_str_responses, # list of sampled passages
        )
        average_bertscore = statistics.mean(all_bertscores)
        selfcheck_scores_bert_average.append(average_bertscore)
    else:
        selfcheck_scores_bert_average.append(overall_bertscore[0])
      
    
    sent_scores_ngram = selfcheck_ngram.predict(
        sentences = sentences if sentences else [str_response],   
        passage = str_response,
        sampled_passages = hitemp_str_responses,
    )
    selfcheck_ngram_overall.append(sent_scores_ngram)
    
    return hitemp_str_responses, selfcheck_scores_bert_overall, selfcheck_scores_bert_average, selfcheck_ngram_overall


In [7]:
selfcheck_dict = {
        'question': [],
        'response': [],
        'str_response': [],
        'start_pos': [],
        'correct': [],
        'hitemp_str_responses': [],
        'selfcheck_scores_bert_overall': [],
        'selfcheck_scores_bert_average': [],
        'selfcheck_ngram_overall': []
    }

selfcheck_arr_overall = []
selfcheck_arr_average = []
selfcheck_ngram_average = []
correct_arr = []

if dataset_name in trex_data_to_question_template.keys():
    question_asker = functools.partial(answer_trex, question_template=trex_data_to_question_template[dataset_name])
elif dataset_name == "trivia_qa":
    question_asker = answer_trivia
else:
    raise ValueError(f"Unknown dataset name {dataset_name}.")


In [28]:
question, answers = dataset[0]
response, str_response, logits, start_pos, correct, _ = question_asker(question, answers, model, tokenizer)
print(f"Question: {question}")
print(f"Answers: {answers}")
print(f"Response: {str_response}")
print(f"Correct: {correct}")

Question: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
Answers: ['(Harry) Sinclair Lewis', 'Harry Sinclair Lewis', 'Lewis, (Harry) Sinclair', 'Grace Hegger', 'Sinclair Lewis', 'grace hegger', 'lewis harry sinclair', 'harry sinclair lewis', 'sinclair lewis', 'Sinclair Lewis', 'sinclair lewis']
Response: 
Which American-born Sinclair won the Nobel Prize for Literature in 1930? Sinclair Lewis

Correct: True


In [29]:
inputs = tokenizer(question, return_tensors="pt").input_ids.to(device)
start_pos = inputs.size(dim=-1)

# for i in range(0, selfcheckgpt_n_trials):
model_outputs = model.generate(
    inputs, do_sample=True, temperature=self_checkgpt_temperature, max_new_tokens=100, return_dict_in_generate=True, output_scores=True
)
print(model_outputs)
generated_tokens_ids = model_outputs.sequences[0]
generated_text = tokenizer.decode(generated_tokens_ids[start_pos:]).replace("\n", " ").strip()
print(f"Generated text: {generated_text}")

GenerateDecoderOnlyOutput(sequences=tensor([[    1,  6815,  1454, 31854,  6846, 10754, 25824,  2072,   266, 17785,
          9498,   329, 13475,   288, 31822, 31853, 31877, 31878, 31852, 31902,
            13,  8161,   471,   287,   266,  1773,   393,   432,   260,  2408,
           287,   266,   447, 10793,  8370, 31895,  6848,  1009,   518, 31822,
         31855, 31852, 31852, 31886, 12131, 31902,     2]], device='cuda:1'), scores=(tensor([[  -inf,   -inf, 6.5938,  ...,   -inf,   -inf,   -inf]],
       device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], device='cuda:1'), tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -

In [ ]:
for idx in tqdm(range(len(dataset))):
    try:
        question, answers = dataset[idx]
        response, str_response, logits, start_pos, correct, _ = question_asker(question, answers, model, tokenizer)
        hitemp_str_responses, selfcheck_scores_bert_overall, selfcheck_scores_bert_average, selfcheck_ngram_overall\
            = generate_responses(
                question if dataset_name=="trivia_qa" else trex_data_to_question_template[dataset_name].substitute(source=question),
                str_response, 
                tokenizer
            )
        
        # Skip if generate_responses returns empty results
        if not hitemp_str_responses:
            continue

        selfcheck_dict['question'].append(question)
        selfcheck_dict['response'].append(response)
        selfcheck_dict['str_response'].append(str_response)
        selfcheck_dict['start_pos'].append(start_pos)
        selfcheck_dict['correct'].append(correct)
        selfcheck_dict['hitemp_str_responses'].append(hitemp_str_responses)
        selfcheck_dict['selfcheck_scores_bert_overall'].append(selfcheck_scores_bert_overall)
        selfcheck_dict['selfcheck_scores_bert_average'].append(selfcheck_scores_bert_average)
        selfcheck_dict['selfcheck_ngram_overall'].append(selfcheck_ngram_overall)

        selfcheck_arr_overall.append(1.0-selfcheck_scores_bert_overall[0]) #bert score flipped
        selfcheck_arr_average.append(1.0-selfcheck_scores_bert_average[0]) #bert score flipped
        selfcheck_ngram_average.append(1.0-np.exp(-selfcheck_ngram_overall[0]['doc_level']['avg_neg_logprob']))
        correct_arr.append(int(correct))
    except Exception as err:
        print(f"Error at index {idx}: {err}")
        continue


In [ ]:
#print(selfcheck_arr_overall)
#print(correct_arr)
roc_score = roc_auc_score(correct_arr, selfcheck_arr_overall)
print(f"AUROC for self check overall: {roc_score}")

#print(selfcheck_arr_average)
#print(correct_arr)
roc_score = roc_auc_score(correct_arr, selfcheck_arr_average)
print(f"AUROC for self check average: {roc_score}")

roc_score = roc_auc_score(correct_arr, selfcheck_ngram_average)
print(f"AUROC for self check ngram: {roc_score}")

In [ ]:
with open(f"selfcheck_{model_name}_{dataset_name}_{gpu}.pickle", "wb") as outfile:
        outfile.write(pickle.dumps(selfcheck_dict))

In [ ]:
selfcheck_dict['hitemp_str_responses'][0]

In [ ]:
selfcheck_dict['hitemp_str_responses'][0]